# DBLP–ACM Identity Resolution

This notebook loads the three DBLP–ACM benchmark CSVs from `@FACEBOOK_STAGE` (either directly or from a ZIP archive), generates candidate paper matches, builds a weighted graph, and prepares results for evaluation.

Run cells in order. The notebook uses the active Snowflake Snowpark session.

In [2]:
from snowflake.snowpark.context import get_active_session
import os
import re
import glob
import zipfile
from collections import defaultdict
from pathlib import Path
import pandas as pd
import networkx as nx
from rapidfuzz.fuzz import ratio

session = get_active_session()
if session is None:
    raise RuntimeError(
        "No active Snowpark session found. Run this notebook inside a Snowflake Python Notebook "
        "or connect it to a Snowflake compute environment."
    )

print("Connected to Snowflake:", session.get_current_database(), session.get_current_schema())

Connected to Snowflake: None None


## 1. List files in the stage

In [3]:
DATABASE_NAME = "GRAPH_ANALYTICS_DB"
SCHEMA_NAME = "FACEBOOK"

session.sql(f'USE DATABASE "{DATABASE_NAME}"').collect()
session.sql(f'USE SCHEMA "{SCHEMA_NAME}"').collect()

stage_files = session.sql("LIST @FACEBOOK_STAGE").collect()

## 2. Download files from the stage and extract the archive if needed

In [4]:
download_dir = "/tmp/dblp_acm_download"
extract_dir = "/tmp/dblp_acm_extracted"

os.makedirs(download_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)

stage_names = [str(row["name"]).split("/")[-1] for row in stage_files]
staged_csv_names = [name for name in stage_names if name.lower().endswith(".csv")]
staged_zip_names = [name for name in stage_names if name.lower().endswith(".zip")]

csv_files = []

if staged_csv_names and not staged_zip_names:  # export patch: prefer DBLP-ACM.zip
    for csv_name in staged_csv_names:
        session.file.get(f"@FACEBOOK_STAGE/{csv_name}", download_dir)
    csv_files = glob.glob(os.path.join(download_dir, "**", "*.csv"), recursive=True)

elif staged_zip_names:
    zip_name = staged_zip_names[0]
    session.file.get(f"@FACEBOOK_STAGE/{zip_name}", download_dir)

    zip_matches = glob.glob(os.path.join(download_dir, "**", zip_name), recursive=True)
    if not zip_matches:
        raise FileNotFoundError(f"Could not find downloaded archive: {zip_name}")

    with zipfile.ZipFile(zip_matches[0], "r") as archive:
        archive.extractall(extract_dir)

    csv_files = glob.glob(os.path.join(extract_dir, "**", "*.csv"), recursive=True)

else:
    raise FileNotFoundError("No .zip or .csv files found in @FACEBOOK_STAGE.")

print("CSV files available:")
for path in csv_files:
    print(os.path.basename(path))

CSV files available:
DBLP-ACM_perfectMapping.csv
ACM.csv
DBLP2.csv


## 4. Load the DBLP and ACM source tables

This cell discovers the source CSVs by filename and deliberately excludes mapping files. It prints the columns so you can verify the benchmark version.

In [5]:
def find_source_file(keyword, paths):
    matches = [
        p for p in paths
        if keyword.lower() in os.path.basename(p).lower()
        and "mapping" not in os.path.basename(p).lower()
        and "perfectmapping" not in os.path.basename(p).lower()
    ]
    if not matches:
        raise FileNotFoundError(
            f"No source CSV containing '{keyword}' found. "
            f"Files found: {[os.path.basename(p) for p in paths]}"
        )
    return matches[0]

dblp_path = find_source_file("dblp", csv_files)
acm_path = find_source_file("acm", csv_files)

try:
    dblp = pd.read_csv(dblp_path, encoding="utf-8-sig")
except UnicodeDecodeError:
    dblp = pd.read_csv(dblp_path, encoding="latin1")
acm = pd.read_csv(acm_path, encoding="utf-8-sig")

print("DBLP file:", os.path.basename(dblp_path), "| shape:", dblp.shape)
print("ACM file:", os.path.basename(acm_path), "| shape:", acm.shape)
print("DBLP columns:", dblp.columns.tolist())
print("ACM columns:", acm.columns.tolist())

display(dblp.head())
display(acm.head())

DBLP file: DBLP2.csv | shape: (2616, 5)
ACM file: ACM.csv | shape: (2294, 5)
DBLP columns: ['id', 'title', 'authors', 'venue', 'year']
ACM columns: ['id', 'title', 'authors', 'venue', 'year']


,id,title,authors,venue,year
0,journals/sigmod/Mackay99,Semantic Integration of Environmental Models f...,D. Scott Mackay,SIGMOD Record,1999
1,conf/vldb/PoosalaI96,Estimation of Query-Result Distribution and it...,"Viswanath Poosala, Yannis E. Ioannidis",VLDB,1996
2,conf/vldb/PalpanasSCP02,Incremental Maintenance for Non-Distributive A...,"Themistoklis Palpanas, Richard Sidle, Hamid Pi...",VLDB,2002
3,conf/vldb/GardarinGT96,Cost-based Selection of Path Expression Proces...,"Zhao-Hui Tang, Georges Gardarin, Jean-Robert G...",VLDB,1996
4,conf/vldb/HoelS95,Benchmarking Spatial Join Operations with Spat...,"Erik G. Hoel, Hanan Samet",VLDB,1995


,id,title,authors,venue,year
0,304586,The WASA2 object-oriented workflow management ...,"Gottfried Vossen, Mathias Weske",International Conference on Management of Data,1999
1,304587,A user-centered interface for querying distrib...,"Isabel F. Cruz, Kimberly M. James",International Conference on Management of Data,1999
2,304589,"World Wide Database-integrating the Web, CORBA...","Athman Bouguettaya, Boualem Benatallah, Lily H...",International Conference on Management of Data,1999
3,304590,XML-based information mediation with MIX,"Chaitan Baru, Amarnath Gupta, Bertram Lud&#228...",International Conference on Management of Data,1999
4,304582,The CCUBE constraint object-oriented database ...,"Alexander Brodsky, Victor E. Segal, Jia Chen, ...",International Conference on Management of Data,1999


## 5. Normalize titles and assign unique graph node IDs

We use title similarity for this first prototype. Later, you can extend matching to authors, venue, and year if those fields are present.

In [6]:
def find_column(df, candidates):
    lookup = {str(c).strip().lower(): c for c in df.columns}
    for name in candidates:
        if name in lookup:
            return lookup[name]
    return None

dblp_title_col = find_column(dblp, ["title"])
acm_title_col = find_column(acm, ["title"])

if dblp_title_col is None or acm_title_col is None:
    raise ValueError(
        "Could not find a 'title' column in both datasets. "
        "Review the column names printed in Cell 4."
    )

def normalize_title(value):
    if pd.isna(value):
        return ""
    value = str(value).lower()
    value = re.sub(r"[^a-z0-9 ]", " ", value)
    return re.sub(r"\\s+", " ", value).strip()

dblp = dblp.reset_index(drop=True).copy()
acm = acm.reset_index(drop=True).copy()

dblp["NODE_ID"] = ["DBLP_" + str(i) for i in range(len(dblp))]
acm["NODE_ID"] = ["ACM_" + str(i) for i in range(len(acm))]

dblp["NORMALIZED_TITLE"] = dblp[dblp_title_col].map(normalize_title)
acm["NORMALIZED_TITLE"] = acm[acm_title_col].map(normalize_title)

print("DBLP records:", len(dblp))
print("ACM records:", len(acm))
display(dblp[["NODE_ID", dblp_title_col, "NORMALIZED_TITLE"]].head())

DBLP records: 2616
ACM records: 2294


,NODE_ID,title,NORMALIZED_TITLE
0,DBLP_0,Semantic Integration of Environmental Models f...,semantic integration of environmental models f...
1,DBLP_1,Estimation of Query-Result Distribution and it...,estimation of query result distribution and it...
2,DBLP_2,Incremental Maintenance for Non-Distributive A...,incremental maintenance for non distributive a...
3,DBLP_3,Cost-based Selection of Path Expression Proces...,cost based selection of path expression proces...
4,DBLP_4,Benchmarking Spatial Join Operations with Spat...,benchmarking spatial join operations with spat...


## 6. Generate candidate pairs

To avoid comparing every DBLP record with every ACM record, index ACM records by title tokens and compare only pairs that share at least one informative token.

In [7]:
STOPWORDS = {
    "the", "and", "for", "with", "from", "using",
    "based", "into", "over", "via", "its"
}

def title_tokens(title):
    return {
        word for word in title.split()
        if len(word) >= 4 and word not in STOPWORDS
    }

token_index = defaultdict(list)
for j, title in enumerate(acm["NORMALIZED_TITLE"]):
    for token in title_tokens(title):
        token_index[token].append(j)

candidate_pairs = set()
for i, title in enumerate(dblp["NORMALIZED_TITLE"]):
    if not title:
        continue
    possible_acm = set()
    for token in title_tokens(title):
        possible_acm.update(token_index.get(token, []))
    for j in possible_acm:
        if acm.at[j, "NORMALIZED_TITLE"]:
            candidate_pairs.add((i, j))

print("Candidate pairs generated:", len(candidate_pairs))

Candidate pairs generated: 666074


## 7. Score candidate matches and create the edges table

The initial threshold is 90/100 title similarity. Treat this as a tunable starting point, not a guaranteed match. Keep scores so you can inspect borderline cases.

In [8]:
MATCH_THRESHOLD = 90

edge_rows = []
for i, j in candidate_pairs:
    title_a = dblp.at[i, "NORMALIZED_TITLE"]
    title_b = acm.at[j, "NORMALIZED_TITLE"]
    score = ratio(title_a, title_b)

    edge_rows.append({
        "SOURCE_ID": dblp.at[i, "NODE_ID"],
        "TARGET_ID": acm.at[j, "NODE_ID"],
        "TITLE_SIMILARITY": float(score),
        "IS_MATCH_CANDIDATE": bool(score >= MATCH_THRESHOLD)
    })

all_candidate_edges_df = pd.DataFrame(edge_rows)
edges_df = all_candidate_edges_df[
    all_candidate_edges_df["IS_MATCH_CANDIDATE"]
].copy() if not all_candidate_edges_df.empty else pd.DataFrame(
    columns=["SOURCE_ID", "TARGET_ID", "TITLE_SIMILARITY", "IS_MATCH_CANDIDATE"]
)

print("All candidate pairs scored:", len(all_candidate_edges_df))
print("Edges above threshold:", len(edges_df))
display(edges_df.sort_values("TITLE_SIMILARITY", ascending=False).head(20))

All candidate pairs scored: 666074
Edges above threshold: 2456


,SOURCE_ID,TARGET_ID,TITLE_SIMILARITY,IS_MATCH_CANDIDATE
665238,DBLP_2550,ACM_1216,100.0,True
127,DBLP_553,ACM_2232,100.0,True
147,DBLP_286,ACM_548,100.0,True
664024,DBLP_2193,ACM_918,100.0,True
393,DBLP_2008,ACM_1236,100.0,True
465,DBLP_1700,ACM_1725,100.0,True
631,DBLP_196,ACM_712,100.0,True
1248,DBLP_1368,ACM_803,100.0,True
1526,DBLP_2412,ACM_1000,100.0,True
1750,DBLP_2444,ACM_969,100.0,True


## 8. Build the graph and inspect connected components

Each source record is a node. An edge means the title similarity passed the threshold. Connected components are proposed clusters, not guaranteed true identities.

In [9]:
G = nx.Graph()

for _, row in dblp.iterrows():
    G.add_node(
        row["NODE_ID"],
        source="DBLP",
        title=str(row[dblp_title_col])
    )

for _, row in acm.iterrows():
    G.add_node(
        row["NODE_ID"],
        source="ACM",
        title=str(row[acm_title_col])
    )

for _, edge in edges_df.iterrows():
    G.add_edge(
        edge["SOURCE_ID"],
        edge["TARGET_ID"],
        weight=edge["TITLE_SIMILARITY"]
    )

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())
print("Connected components:", nx.number_connected_components(G))
print("Nodes with at least one edge:", sum(1 for _, d in G.degree() if d > 0))

Graph nodes: 4910
Graph edges: 2456
Connected components: 2684
Nodes with at least one edge: 4322


## 9. Assign cluster IDs

Isolated nodes are included as singleton clusters. For evaluation, we will focus on predicted DBLP–ACM pairs rather than assuming every component is correct.

In [10]:
cluster_rows = []
for cluster_num, component in enumerate(nx.connected_components(G), start=1):
    for node_id in component:
        cluster_rows.append({
            "NODE_ID": node_id,
            "CLUSTER_ID": cluster_num,
            "SOURCE": G.nodes[node_id]["source"],
            "TITLE": G.nodes[node_id]["title"],
            "DEGREE": G.degree(node_id)
        })

clusters_df = pd.DataFrame(cluster_rows)

non_singleton_ids = (
    clusters_df.groupby("CLUSTER_ID")["NODE_ID"].count()
    .loc[lambda s: s > 1].index
)
display(
    clusters_df[clusters_df["CLUSTER_ID"].isin(non_singleton_ids)]
    .sort_values(["CLUSTER_ID", "SOURCE"])
    .head(100)
)

,NODE_ID,CLUSTER_ID,SOURCE,TITLE,DEGREE
0,ACM_117,1,ACM,Semantic integration of environmental models f...,1
1,DBLP_0,1,DBLP,Semantic Integration of Environmental Models f...,1
2,ACM_1093,2,ACM,Estimation of Query-Result Distribution and it...,1
3,DBLP_1,2,DBLP,Estimation of Query-Result Distribution and it...,1
6,ACM_1125,4,ACM,Cost-based Selection of Path Expression Proces...,1
...,...,...,...,...,...
99,DBLP_51,52,DBLP,Multimedia streaming in large-scale sensor net...,1
102,ACM_1680,53,ACM,How to modify SQL queries in order to guarante...,1
101,DBLP_52,53,DBLP,How to Modify SQL Queries in Order to Guarante...,1
103,ACM_1433,54,ACM,Materialization: A Powerful and Ubiquitous Abs...,1


## 10. Load the ground-truth mapping file

The mapping is for evaluation only. Do not use it to generate candidate pairs or select the similarity threshold. First inspect its column names and sample rows.

In [11]:
mapping_files = [
    p for p in csv_files
    if "mapping" in os.path.basename(p).lower()
]

if not mapping_files:
    raise FileNotFoundError(
        "No mapping CSV found in extracted files. "
        "Check the CSV filenames printed in Cell 3."
    )

mapping_path = mapping_files[0]
truth = pd.read_csv(mapping_path, encoding="utf-8-sig")

print("Mapping file:", os.path.basename(mapping_path))
print("Ground-truth shape:", truth.shape)
print("Ground-truth columns:", truth.columns.tolist())
display(truth.head(10))

Mapping file: DBLP-ACM_perfectMapping.csv
Ground-truth shape: (2224, 2)
Ground-truth columns: ['idDBLP', 'idACM']


,idDBLP,idACM
0,conf/sigmod/SlivinskasJS01,375678
1,conf/sigmod/ChaudhuriDN01,375694
2,conf/sigmod/RinfretOO01,375669
3,conf/sigmod/BreunigKKS01,375672
4,conf/sigmod/JagadishJOT01,375687
5,conf/sigmod/NazeriBO01,375743
6,conf/sigmod/ChenZCKR01,375789
7,conf/sigmod/BohmBKK01,375714
8,conf/sigmod/DattaDRTV01,375780
9,conf/sigmod/HernandezMHYHT01,375767


## 10b. Evaluate against the ground truth

Precision, recall, and F1 at two levels: **edges** (pairs above the threshold) and **clusters** (every DBLP–ACM pair that ends up in the same connected component, including transitive links).

In [12]:
dblp_id_col = find_column(dblp, ["id"])
acm_id_col = find_column(acm, ["id"])

node_to_id = {
    **dict(zip(dblp["NODE_ID"], dblp[dblp_id_col].astype(str).str.strip())),
    **dict(zip(acm["NODE_ID"], acm[acm_id_col].astype(str).str.strip())),
}

truth_pairs = set(zip(
    truth["idDBLP"].astype(str).str.strip(),
    truth["idACM"].astype(str).str.strip()
))

def prf(predicted, actual):
    tp = len(predicted & actual)
    fp = len(predicted - actual)
    fn = len(actual - predicted)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"TP": tp, "FP": fp, "FN": fn,
            "PRECISION": round(precision, 4), "RECALL": round(recall, 4), "F1": round(f1, 4)}

# Edge level
edge_pairs = {
    (node_to_id[s], node_to_id[t])
    for s, t in zip(edges_df["SOURCE_ID"], edges_df["TARGET_ID"])
}

# Cluster level: all cross-source pairs within each component
cluster_pairs = set()
for _, grp in clusters_df[clusters_df["CLUSTER_ID"].isin(non_singleton_ids)].groupby("CLUSTER_ID"):
    d_ids = [node_to_id[n] for n in grp.loc[grp["SOURCE"] == "DBLP", "NODE_ID"]]
    a_ids = [node_to_id[n] for n in grp.loc[grp["SOURCE"] == "ACM", "NODE_ID"]]
    cluster_pairs.update((d, a) for d in d_ids for a in a_ids)

eval_df = pd.DataFrame([
    {"LEVEL": "EDGES", "PREDICTED": len(edge_pairs), **prf(edge_pairs, truth_pairs)},
    {"LEVEL": "CLUSTERS", "PREDICTED": len(cluster_pairs), **prf(cluster_pairs, truth_pairs)},
])
print("Ground-truth pairs:", len(truth_pairs))
print("Ground-truth IDs found in source data:",
      sum(d in node_to_id.values() for d, _ in truth_pairs))
display(eval_df)

# Threshold sweep (edge level)
sweep = []
for t in range(80, 101, 2):
    sub = all_candidate_edges_df[all_candidate_edges_df["TITLE_SIMILARITY"] >= t]
    pairs = {(node_to_id[s], node_to_id[x]) for s, x in zip(sub["SOURCE_ID"], sub["TARGET_ID"])}
    sweep.append({"THRESHOLD": t, "PREDICTED": len(pairs), **prf(pairs, truth_pairs)})
display(pd.DataFrame(sweep))

Ground-truth pairs: 2224
Ground-truth IDs found in source data: 2224


,LEVEL,PREDICTED,TP,FP,FN,PRECISION,RECALL,F1
0,EDGES,2456,2142,314,82,0.8721,0.9631,0.9154
1,CLUSTERS,2457,2142,315,82,0.8718,0.9631,0.9152


,THRESHOLD,PREDICTED,TP,FP,FN,PRECISION,RECALL,F1
0,80,2552,2177,375,47,0.8531,0.9789,0.9116
1,82,2526,2171,355,53,0.8595,0.9762,0.9141
2,84,2500,2165,335,59,0.8660,0.9735,0.9166
3,86,2480,2154,326,70,0.8685,0.9685,0.9158
4,88,2468,2149,319,75,0.8707,0.9663,0.9160
5,90,2456,2142,314,82,0.8721,0.9631,0.9154
6,92,2431,2124,307,100,0.8737,0.9550,0.9126
7,94,2411,2109,302,115,0.8747,0.9483,0.9100
8,96,2399,2101,298,123,0.8758,0.9447,0.9089
9,98,2339,2072,267,152,0.8858,0.9317,0.9082


## 11. Save outputs as Snowflake tables

This saves the raw records, candidate edges above threshold, and cluster assignments into a schema you can access. Change the target schema if you prefer. `write_pandas` requires an appropriate Snowflake connector setup; Snowpark's `Session.write_pandas` is also available in many notebook runtimes.

## Notes and next improvements
- **Mapping evaluation:** The mapping file's column names and ID conventions vary by dataset release. Use Cell 10's output to build a version-specific join and compute precision, recall, and F1. Do not guess mapping ID formats.
- **Improve candidate generation:** Shared-token blocking can miss pairs when titles are heavily corrupted or have no shared long token. Consider multiple blocking keys and character n-gram similarity.
- **Use more fields:** If present, combine title similarity with author, venue, and year agreement.
- **Avoid naive transitive merges:** Connected components can join unrelated records through a false edge. Inspect large clusters and consider one-to-one matching constraints for this two-source benchmark.
- **Production use:** Treat this as a prototype. For robust identity resolution, validate the threshold against a held-out subset of known pairs and inspect false positives and false negatives.